# Download Florida Demographics

Downloads racial demographic data from the Census API (ACS 5-year 2023) and saves as JSON files.

**Requirements:**
```bash
pip install requests python-dotenv
```

**Setup:**
Create a `.env` file in the project root with:
```
census_api_key=YOUR_API_KEY_HERE
```
Get a free key at: https://api.census.gov/data/key_signup.html


In [6]:
import os
import json
import requests
from pathlib import Path
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
census_api_key = os.getenv("census_api_key")

if not census_api_key:
    raise ValueError("Missing census_api_key in .env file!")

# Output directory
PUBLIC_DIR = (Path.cwd().parent / "public").resolve()
print(f"Output directory: {PUBLIC_DIR}")

# Florida FIPS code
STATE_FIPS = "12"

# Census API base URL
API_URL = "https://api.census.gov/data/2023/acs/acs5"

# Variables to fetch (ACS table B03002 = Hispanic/Latino Origin by Race)
VARIABLES = {
    "B03002_001E": "total_pop",
    "B03002_003E": "white_non_hisp",
    "B03002_004E": "black_non_hisp",
    "B03002_005E": "native_am_non_hisp",
    "B03002_006E": "asian_non_hisp",
    "B03002_012E": "hisp_any_race",
    "B03002_013E": "white_hisp",
}


Output directory: /Users/peter/familymaps/public


In [7]:
# ============================================================
# COUNTY DEMOGRAPHICS
# ============================================================
print("Downloading county demographics...")

response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "county:*",
    "in": f"state:{STATE_FIPS}",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

# Convert to list of dicts
county_data = []
headers = data[0]
for row in data[1:]:
    record = {"GEOID": row[headers.index("state")] + row[headers.index("county")]}
    record["name"] = row[headers.index("NAME")]
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        record[var_name] = int(val) if val and val != "-666666666" else 0
    county_data.append(record)

# Save to JSON
with open(PUBLIC_DIR / "FL_county_demographics.json", "w") as f:
    json.dump(county_data, f)

print(f"  ✅ Saved {len(county_data)} counties to FL_county_demographics.json")


  ✅ Saved 67 counties to FL_county_demographics.json


In [8]:
# ============================================================
# TRACT DEMOGRAPHICS
# ============================================================
print("Downloading tract demographics...")

response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "tract:*",
    "in": f"state:{STATE_FIPS} county:*",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

# Convert to list of dicts
tract_data = []
headers = data[0]
for row in data[1:]:
    record = {"GEOID": row[headers.index("state")] + row[headers.index("county")] + row[headers.index("tract")]}
    record["name"] = row[headers.index("NAME")]
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        record[var_name] = int(val) if val and val != "-666666666" else 0
    tract_data.append(record)

# Save to JSON
with open(PUBLIC_DIR / "FL_tract_demographics.json", "w") as f:
    json.dump(tract_data, f)

print(f"  ✅ Saved {len(tract_data)} tracts to FL_tract_demographics.json")


  ✅ Saved 5160 tracts to FL_tract_demographics.json


In [9]:
# ============================================================
# BLOCK GROUP DEMOGRAPHICS
# ============================================================
print("Downloading block group demographics...")

response = requests.get(API_URL, params={
    "get": "NAME," + ",".join(VARIABLES.keys()),
    "for": "block group:*",
    "in": f"state:{STATE_FIPS} county:*",
    "key": census_api_key,
})
response.raise_for_status()
data = response.json()

# Convert to list of dicts
bg_data = []
headers = data[0]
for row in data[1:]:
    record = {"GEOID": row[headers.index("state")] + row[headers.index("county")] + row[headers.index("tract")] + row[headers.index("block group")]}
    record["name"] = row[headers.index("NAME")]
    for var_code, var_name in VARIABLES.items():
        val = row[headers.index(var_code)]
        record[var_name] = int(val) if val and val != "-666666666" else 0
    bg_data.append(record)

# Save to JSON
with open(PUBLIC_DIR / "FL_block_group_demographics.json", "w") as f:
    json.dump(bg_data, f)

print(f"  ✅ Saved {len(bg_data)} block groups to FL_block_group_demographics.json")


  ✅ Saved 13388 block groups to FL_block_group_demographics.json


In [10]:
# ============================================================
# DONE - Show sample output
# ============================================================
print("\n✅ All done! Created:")
print("  - FL_county_demographics.json")
print("  - FL_tract_demographics.json")
print("  - FL_block_group_demographics.json")

print("\nSample county record:")
print(json.dumps(county_data[0], indent=2))



✅ All done! Created:
  - FL_county_demographics.json
  - FL_tract_demographics.json
  - FL_block_group_demographics.json

Sample county record:
{
  "GEOID": "12001",
  "name": "Alachua County, Florida",
  "total_pop": 281751,
  "white_non_hisp": 164301,
  "black_non_hisp": 53662,
  "native_am_non_hisp": 270,
  "asian_non_hisp": 16444,
  "hisp_any_race": 34392,
  "white_hisp": 12309
}
